In [ ]:
import numpy as np
from lib.dvr2d import *
from lib.utils import *
from lib.vqe import DVR_VQE
from lib.pot_gen import get_pot

import matplotlib.pyplot as plt
from matplotlib import rc

log_file = 'results/log.txt'

cmap = plt.get_cmap('tab10')
plt.rcParams["figure.figsize"] = (5,5)
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 300
rc('text', usetex=True)
rc('font',**{'family':'serif','serif':['Computer Modern']})
%load_ext autoreload

In [ ]:
mol_params = mgnh_params
pot_file = 'mgnh.txt'

dvr_options = {
    'type': 'jacobi',
    'N_R': 20, 
    'N_theta': 4,
    'l': 1,
    'K_max': 0,
    'r_min': 3,
    'r_max': 6, 
    'trunc': 0,
    'J': 0
}

vqe_options = {
    'optimizers': ['L_BFGS_B.8000'],
    # 'optimizers': ['COBYLA.100'],
    # 'repeats': 1, 
    'repeats': 5, 
    'beta': [1000 for i in range(10)],
    'seed': 42
}

ansatz_options = {
    'type': 'twolocal',
    'entanglement': 'opt',
    'gates': [9, 4, 1],
    'reps': 1,
}


In [ ]:
import qiskit
import qiskit_algorithms
import qiskit_aer

print(qiskit.__version__)
print(qiskit_algorithms.__version__)
print(qiskit_aer.__version__)

In [ ]:
pot = get_pot(pot_file, take_cos=False)
dvr_vqe = DVR_VQE(mol_params, pot, log_dir='results/tests/')
# opt_params = dvr_vqe.get_dvr_vqe(dvr_options, ansatz_options, vqe_options, excited_states=True)

In [ ]:
ansatz_options2 = {
    'type': 'twolocal',
    'entanglement': 'opt',
    'gates': [39, 4, 21, 8, 13, 19, 7, 34, 30, 14],
    'reps': 4,
}

# opt_params2 = dvr_vqe.get_dvr_vqe(dvr_options, ansatz_options2, vqe_options, excited_states=True)

In [ ]:
with np.load('results/excited_states/mgnh_twolocal_opt_1/opt_params.npz', allow_pickle=True) as data:
    opt_params = data['opt_params']

with np.load('results/excited_states/mgnh_twolocal_opt_4/opt_params.npz', allow_pickle=True) as data:
    opt_params2 = data['opt_params']
# opt_params = []
# energies = []
# with np.load('results/excited_states/mgnh_twolocal_opt_1(1)/vqe.npz', allow_pickle=True) as data:
#     opt_params.append(*data['params'])
#     energies.append(data['energies'])

# for i in range(1, 11):
#     with np.load(f'results/excited_states/mgnh_twolocal_opt_1(1)/vqe_excited{i}.npz', allow_pickle=True) as data:
#         opt_params.append(*data['params'])
#         energies.append(data['energies'])

# energies = np.array(energies).flatten()
# opt_params = np.array(opt_params)
# print(opt_params.shape)

# opt_params2 = []
# energies2 = []
# with np.load('results/excited_states/mgnh_twolocal_opt_4/vqe.npz', allow_pickle=True) as data:
#     opt_params2.append(*data['params'])
#     energies2.append(data['energies'])

# for i in range(1, 11):
#     with np.load(f'results/excited_states/mgnh_twolocal_opt_4/vqe_excited{i}.npz', allow_pickle=True) as data:
#         opt_params2.append(*data['params'])
#         energies2.append(data['energies'])

# energies2 = np.array(energies2).flatten()
# opt_params2 = np.array(opt_params2)
# print(opt_params2.shape)

In [ ]:
ansatz = dvr_vqe.get_ansatz(ansatz_options, 5)
ansatz2 = dvr_vqe.get_ansatz(ansatz_options2, 5)
print(ansatz)
print(ansatz2)

In [ ]:
circs = []
for i in range(11):
    circ = ansatz.assign_parameters(opt_params[i], inplace=False)
    circs.append(circ)

print(circs[0])

In [ ]:
circs2 = []
for i in range(11):
    circ = ansatz2.assign_parameters(opt_params2[i], inplace=False)
    circs2.append(circ)

print(circs2[0])

In [ ]:
from qiskit.quantum_info import Statevector

sv1s = np.array([Statevector.from_instruction(circ) for circ in circs])
sv2s = np.array([Statevector.from_instruction(circ) for circ in circs2])

In [ ]:
h_dvr = dvr_vqe.get_h_dvr(dvr_options, J=0) * hartree
e_vals, e_vecs = np.linalg.eigh(h_dvr)
print(e_vals[:5])

In [ ]:
i = 4
plt.plot(e_vecs[:,i])
plt.plot(sv2s[i] * np.sign(sv2s[i] * e_vecs[:,i]))

In [ ]:
sv1s_new = []
sv2s_new = []
for i in range(11):
    sv1s_new.append(sv1s[i] * np.sign(sv1s[i] * e_vecs[:,i]))
    sv2s_new.append(sv2s[i] * np.sign(sv2s[i] * e_vecs[:,i]))

sv1s_new = np.array(sv1s_new)
sv2s_new = np.array(sv2s_new)

In [ ]:
fid1s = []
for i in range(11):
    fid1 = np.abs(np.dot(sv1s_new[i], e_vecs[:,i].conj()))**2
    fid1s.append(fid1)
    print(f'Fidelity VQE 1 state {i}: {fid1}')
print('----')
fid2s = []
for i in range(10):
    fid2 = np.abs(np.dot(sv2s_new[i], e_vecs[:,i].conj()))**2
    fid2s.append(fid2)
    print(f'Fidelity VQE 2 state {i}: {fid2}')

In [ ]:
print(sv2s_new[0].conj().T @ h_dvr @ sv2s_new[0])
print(e_vecs[:, 0].conj().T @ h_dvr @ e_vecs[:, 0])

In [ ]:
plt.rcParams["figure.figsize"] = (5,5)
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 300

ax = plt.subplot()
ax.plot(fid1s[:-1], 'o-', label='$\\mathcal{C}_1$')
ax.plot(fid2s, 'x-', label='$\\mathcal{C}_{0.01}$')
ax.spines.right.set_visible(False)
ax.spines.top.set_visible(False)
ax.set_xlabel('$\\nu$')
ax.set_ylabel('Fidelity')
ax.legend()
plt.ylim(0, 1.05)
plt.savefig('figs/mgnh_fidelity.pdf', bbox_inches='tight')